# CP-SAT Scheduler Example

## Introduction

COASTSim has three schedulers for building a plan ahead of time. All three
maximize the same thing, merit-weighted science (each snapshot's merit times its
science seconds), and every plan they return executes exactly as planned:

* `PriorityScheduler` places requests in priority order, each at the earliest time
  it fits. It is fast but never revisits a decision.
* `LocalSearchScheduler` starts from the priority-first plan and improves it by
  trying many small changes to the order of its snapshots.
* `CpSatScheduler` starts from the same plan, then splits the horizon into chunks
  (three hours by default) and, for each, lets Google's OR-Tools CP-SAT
  constraint solver choose which snapshots to observe and in what order. Each
  chunk's result is checked with the same exact rules and kept only if it is at
  least as good as the priority-first plan's snapshots there. It can then spend
  more time improving the result by local search.

This notebook compares CP-SAT with local search, using the priority-first plan as
the baseline:

1. on a single one-day plan, with each given the same planning time;
2. chunk by chunk, to see where the solver finds its gains;
3. as the planning time grows;
4. in rolling-horizon simulations over several random sets of targets.

`CpSatScheduler` needs OR-Tools: `pip install "coast-sim[cpsat]"`.

Both schedulers run to a time limit here, so the exact numbers vary a little from
run to run and from machine to machine. The notebook takes about 15 minutes to
run.

## Step 1: Import Libraries

In [ ]:
%matplotlib inline
import time
from datetime import datetime, timedelta, timezone

import matplotlib.pyplot as plt
import numpy as np
from rust_ephem import TLEEphemeris

from conops import DITL, RollingHorizonDITL, create_ditl
from conops.common import ACSMode
from conops.config import (
    MissionConfig,
    ReplanSettings,
    SchedulerConfig,
    Telescope,
    TelescopeConfig,
    TelescopeType,
)
from conops.schedulers import CpSatScheduler, LocalSearchScheduler, PriorityScheduler
from conops.targets import Pointing

## Step 2: Spacecraft Configuration and Ephemeris

The same setup as the Rolling-Horizon DITL and Local Search Scheduler examples: the
default `MissionConfig` with a small telescope, and a one-day ephemeris from a
TLE.

A DITL attaches the ephemeris to the star trackers and radiators when it is
created. Here we call the schedulers directly, before any DITL exists, so we attach
it ourselves.

In [ ]:
config = MissionConfig()
config.payload.instruments.append(
    Telescope(
        name="Telescope f/7 0.5m",
        optics=TelescopeConfig(
            aperture_m=0.5,
            focal_length_m=3.5,
            telescope_type=TelescopeType.RITCHEY_CHRETIEN,
            tube_length_m=1.0,
        ),
    )
)
config.constraint.ignore_roll = True
config.spacecraft_bus.attitude_control.slew_before_visible = False

begin = datetime(2025, 12, 1, tzinfo=timezone.utc)
end = begin + timedelta(days=1)
ephemeris = TLEEphemeris(begin=begin, end=end, step_size=60, tle="example.tle")
config.constraint.ephem = ephemeris
config.spacecraft_bus.star_trackers.set_ephem(ephemeris)
config.spacecraft_bus.radiators.set_ephem(ephemeris)

## Step 3: Create the Targets

100 targets at random positions, each needing 2000 s of exposure in snapshots of
300 to 1000 s, with merit between 10 and 100: about 55 hours of exposure asked
for in a 24-hour day.

The schedulers don't modify the targets they are given, but a simulation credits
collected science to them, so `make_targets` builds a fresh set for each run.

In [ ]:
def make_targets(seed: int) -> list[Pointing]:
    rng = np.random.default_rng(seed)
    targets = []
    for i in range(100):
        target = Pointing(
            config=config,
            ra=rng.uniform(0, 360),
            dec=rng.uniform(-90, 90),
            obsid=10000 + i,
            name=f"Target {i}",
            merit=rng.uniform(10, 100),
            ss_min=300,
            ss_max=1000,
        )
        target.exptime = 2000
        targets.append(target)
    return targets


targets = make_targets(5)

## Step 4: Build a One-Day Plan with Each Scheduler

Each scheduler gets the same 40 seconds of planning time on top of building the
priority-first plan:

* **Local search**: `time_limit=40`, all of it searching.
* **CP-SAT**: `solver_time_limit=40`, shared across the day's eight chunks.
* **CP-SAT + local search**: `solver_time_limit=30`, then `time_limit=10` of
  local search starting from the solver's plan.

`CpSatScheduler`'s main settings:

* `solver_time_limit`: seconds the solver may search, shared across the chunks.
* `chunk`: the length of each chunk (default three hours).
* `workers`: the solver's parallel search workers (default 8). Use 1, with
  `seed`, for runs that repeat exactly.
* `time_limit`: seconds of local search afterwards (default 0, none).

After planning, `solver_statuses` gives the solver's result for each chunk
(`OPTIMAL` if it proved its answer the best for the chunk, `FEASIBLE` if it found
one but ran out of time to prove it), and `solver_chunks_used` whether that
chunk's solution was kept.

In [ ]:
BUDGET = 40.0

SCHEDULERS = {
    "Priority": (PriorityScheduler, {}),
    "Local search": (LocalSearchScheduler, {"time_limit": BUDGET, "seed": 1}),
    "CP-SAT": (CpSatScheduler, {"solver_time_limit": BUDGET, "seed": 1}),
    "CP-SAT + local search": (
        CpSatScheduler,
        {"solver_time_limit": 30.0, "time_limit": 10.0, "seed": 1},
    ),
}

# One colour per scheduler in every plot.
COLORS = {
    "Priority": "tab:grey",
    "Local search": "tab:blue",
    "CP-SAT": "tab:orange",
    "CP-SAT + local search": "tab:green",
}

schedulers = {}
plans = {}
planning_seconds = {}
for label, (cls, options) in SCHEDULERS.items():
    start = time.perf_counter()
    scheduler = cls(config, targets, begin, end, **options)
    plans[label] = scheduler.schedule()
    planning_seconds[label] = time.perf_counter() - start
    schedulers[label] = scheduler
    print(f"{label:<22} {planning_seconds[label]:5.1f} s")

cp_sat = schedulers["CP-SAT"]
print()
print("CP-SAT chunk statuses:", cp_sat.solver_statuses)
print("CP-SAT chunks kept:   ", cp_sat.solver_chunks_used)

## Step 5: Execute the Plans

Each plan is executed by a `DITL` with the configuration, ephemeris, horizon and
step size it was built with. `validate_plan_matches_execution()` returns an empty
list when the simulation did exactly what the plan said.

In [ ]:
def execute(plan, scheduler) -> DITL:
    ditl = DITL(config=config, plan=plan, ephem=ephemeris, begin=begin, end=end)
    ditl.step_size = scheduler.ctx.step_size
    ditl.calc()
    return ditl


runs = {label: execute(plans[label], schedulers[label]) for label in SCHEDULERS}
for label, run in runs.items():
    print(f"{label:<22} mismatches: {run.validate_plan_matches_execution()}")

## Step 6: Compare the Science

For each plan:

* **science**: hours of science collected;
* **merit-weighted**: each observation's merit times its science hours, summed,
  which is what every scheduler maximizes;
* **gain**: merit-weighted science gained over the priority-first plan;
* **idle**: the share of the day the spacecraft spent idle;
* **observations**, **targets observed** and **targets completed**.

Each scheduler maximizes merit-weighted science, which does not reward finishing a
target. CP-SAT gets the most science, but spreads it over more targets than the
priority-first plan does, so it can complete fewer of them.

In [ ]:
def science_hours(run) -> float:
    return sum(hk.collection_seconds for hk in run.telemetry.housekeeping) / 3600


def merit_hours(run, begin_time=None, end_time=None) -> float:
    """Merit-weighted science hours, optionally only between two Unix times."""
    total = 0.0
    for e in run.plan:
        if e.obstype.name != "AT":
            continue
        lo = e.begin if begin_time is None else max(e.begin, begin_time)
        hi = e.end if end_time is None else min(e.end, end_time)
        if hi > lo:
            total += e.merit * e.collection_seconds_between(lo, hi)
    return total / 3600


def idle_percent(run) -> float:
    return 100 * np.mean(np.asarray(run.mode) == ACSMode.IDLE)


def seconds_by_target(run) -> dict[int, float]:
    seconds: dict[int, float] = {}
    for e in run.plan:
        if e.obstype.name == "AT":
            collected = e.collection_seconds_between(e.begin, e.end)
            seconds[e.obsid] = seconds.get(e.obsid, 0.0) + collected
    return seconds


baseline = merit_hours(runs["Priority"])
print(
    f"{'scheduler':<22} {'science':>8} {'merit-weighted':>15} {'gain':>6} {'idle':>6} "
    f"{'observations':>12} {'observed':>8} {'completed':>9} {'time':>6}"
)
for label, run in runs.items():
    per_target = seconds_by_target(run)
    observations = sum(1 for e in run.plan if e.obstype.name == "AT")
    completed = sum(1 for s in per_target.values() if s >= 2000)
    gain = 100 * (merit_hours(run) / baseline - 1)
    print(
        f"{label:<22} {science_hours(run):7.2f}h {merit_hours(run):15.0f} "
        f"{gain:5.1f}% {idle_percent(run):5.1f}% {observations:12d} "
        f"{len(per_target):8d} {completed:9d} {planning_seconds[label]:5.1f}s"
    )

## Step 7: Where the Time Goes

The share of the day each plan spends in each attitude-control mode.

In [ ]:
MODE_COLORS = {
    ACSMode.SCIENCE: "tab:green",
    ACSMode.SLEWING: "tab:blue",
    ACSMode.PASS: "tab:orange",
    ACSMode.SAA: "tab:purple",
    ACSMode.CHARGING: "tab:olive",
    ACSMode.SAFE: "black",
    ACSMode.IDLE: "tab:red",
}
used_modes = [m for m in MODE_COLORS if any(m in run.mode for run in runs.values())]

fig, ax = plt.subplots(figsize=(9, 3))
for row, run in enumerate(runs.values()):
    modes = np.asarray(run.mode)
    left = 0.0
    for mode in used_modes:
        share = 100 * np.mean(modes == mode)
        ax.barh(
            row,
            share,
            left=left,
            color=MODE_COLORS[mode],
            label=mode.name if row == 0 else None,
        )
        if share > 3:
            ax.text(
                left + share / 2,
                row,
                f"{share:.1f}%",
                ha="center",
                va="center",
                color="white",
                fontsize=9,
            )
        left += share
ax.set_yticks(range(len(runs)), list(runs))
ax.invert_yaxis()
ax.set_xlim(0, 100)
ax.set_xlabel("Share of the day (%)")
ax.legend(
    ncol=len(used_modes), loc="upper center", bbox_to_anchor=(0.5, -0.3), frameon=False
)
fig.tight_layout()
plt.show()

## Step 8: The Timelines

The spacecraft's mode through the day for each plan, with hours 8 to 12 enlarged
below. Red marks idle time. The priority-first plan leaves a long idle gap a
little after 8 hours, when nothing it had left could be reached; local search
keeps it, and CP-SAT fills it.

In [ ]:
def mode_runs(run):
    """Yield (mode, start hour, length in hours) for each run of one mode."""
    modes = np.asarray(run.mode)
    hours = (np.asarray(run.utime) - run.utime[0]) / 3600
    step = run.step_size / 3600
    changes = np.flatnonzero(np.diff(modes)) + 1
    for first, last in zip(np.r_[0, changes], np.r_[changes, len(modes)]):
        yield ACSMode(modes[first]), hours[first], (last - first) * step


def plot_timelines(ax, xlim):
    for row, run in enumerate(runs.values()):
        for mode, start, length in mode_runs(run):
            ax.broken_barh([(start, length)], (row - 0.4, 0.8), color=MODE_COLORS[mode])
    ax.set_yticks(range(len(runs)), list(runs))
    ax.invert_yaxis()
    ax.set_xlim(*xlim)
    ax.set_xlabel("Hours since start of day")


fig, (full, zoom) = plt.subplots(2, 1, figsize=(10, 6.5))
plot_timelines(full, (0, 24))
plot_timelines(zoom, (8, 12))
full.set_title("Whole day")
zoom.set_title("Hours 8 to 12")
handles = [plt.Rectangle((0, 0), 1, 1, color=MODE_COLORS[m]) for m in used_modes]
fig.legend(
    handles,
    [m.name for m in used_modes],
    ncol=len(used_modes),
    loc="lower center",
    frameon=False,
)
fig.tight_layout(rect=(0, 0.05, 1, 1))
plt.show()

## Step 9: Chunk by Chunk

CP-SAT solves the day in three-hour chunks. Here is the merit-weighted science
each plan collects in each chunk, as a gain over the priority-first plan, with
whether the solver's solution for the chunk was kept.

A chunk's gain does not come only from its own solution. A snapshot whose slew
starts near the end of a chunk can finish in the next one, and each chunk starts
from wherever the one before left the spacecraft, with whatever exposure it left.
So a chunk whose solution was not kept can still differ from the priority-first
plan, and a gain in one chunk can come with a small loss in another.

In [ ]:
chunk_hours = cp_sat.chunk.total_seconds() / 3600
edges = np.arange(0, 24 + chunk_hours / 2, chunk_hours)
t0 = begin.timestamp()


def per_chunk(run) -> np.ndarray:
    return np.array(
        [
            merit_hours(run, t0 + lo * 3600, t0 + hi * 3600)
            for lo, hi in zip(edges, edges[1:])
        ]
    )


priority_chunks = per_chunk(runs["Priority"])
others = [label for label in runs if label != "Priority"]
x = np.arange(len(edges) - 1)
width = 0.8 / len(others)

fig, ax = plt.subplots(figsize=(10, 3.8))
for n, label in enumerate(others):
    ax.bar(
        x + (n - (len(others) - 1) / 2) * width,
        per_chunk(runs[label]) - priority_chunks,
        width,
        label=label,
        color=COLORS[label],
    )
ax.axhline(0, color="grey", lw=0.8)
for k, kept in enumerate(cp_sat.solver_chunks_used):
    ax.annotate(
        "kept" if kept else "not kept",
        (k, 1.0),
        xycoords=("data", "axes fraction"),
        ha="center",
        va="top",
        fontsize=8,
        color="dimgrey",
    )
ax.set_xticks(x, [f"{lo:.0f}-{hi:.0f} h" for lo, hi in zip(edges, edges[1:])])
ax.set_ylabel("Merit-weighted science gained\n(merit-hours)")
ax.set_title("Gain over the priority-first plan, by chunk", pad=18)
ax.legend(fontsize=8)
fig.tight_layout()
plt.show()

## Step 10: Merit-Weighted Science Through the Day

The running total of merit-weighted science for each plan, the priority-first
plan included (top), and how far each plan is ahead of the priority-first plan
(bottom).

In [ ]:
hours = np.linspace(0, 24, 289)


def cumulative(run) -> np.ndarray:
    return np.array([merit_hours(run, None, t0 + h * 3600) for h in hours])


curves = {label: cumulative(run) for label, run in runs.items()}

fig, (total_ax, ahead_ax) = plt.subplots(
    2, 1, figsize=(9, 6.5), sharex=True, gridspec_kw={"height_ratios": [3, 2]}
)
for label, curve in curves.items():
    total_ax.plot(hours, curve, label=label, color=COLORS[label])
    ahead_ax.plot(hours, curve - curves["Priority"], label=label, color=COLORS[label])
total_ax.set_ylabel("Merit-weighted science\n(merit-hours)")
total_ax.legend(fontsize=8)
ahead_ax.set_ylabel("Ahead of priority-first\n(merit-hours)")
for ax in (total_ax, ahead_ax):
    for edge in edges[1:-1]:
        ax.axvline(edge, color="grey", lw=0.5, ls=":")
ahead_ax.set_xlim(0, 24)
ahead_ax.set_xlabel("Hours since start of day (dotted lines: CP-SAT chunks)")
fig.tight_layout()
plt.show()

## Step 11: Which Targets Gained

The change in planned exposure for each target, from the priority-first plan, for
local search and for CP-SAT, with targets in order of merit. Both move time
between targets when that fits more science in overall; CP-SAT rearranges much
more of the plan, and gives time to many targets just below those the
priority-first plan observes.

In [ ]:
by_merit = sorted(targets, key=lambda t: t.merit, reverse=True)
priority_by_target = seconds_by_target(runs["Priority"])

fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True, sharey=True)
for ax, label in zip(axes, ("Local search", "CP-SAT")):
    by_target = seconds_by_target(runs[label])
    change = np.array(
        [
            by_target.get(t.obsid, 0.0) - priority_by_target.get(t.obsid, 0.0)
            for t in by_merit
        ]
    )
    ax.bar(
        range(len(by_merit)),
        change / 60,
        color=np.where(change >= 0, "tab:green", "tab:red"),
    )
    ax.axhline(0, color="grey", lw=0.8)
    ax.set_ylabel("Change (min)")
    ax.set_title(
        f"{label}: {np.sum(change > 0)} targets gain time, {np.sum(change < 0)} lose"
    )
axes[-1].set_xlabel("Targets, highest merit first")
axes[-1].set_xlim(-1, len(by_merit))
fig.tight_layout()
plt.show()

## Step 12: More Planning Time

We build the plan again with different time budgets, giving local search and
CP-SAT the same time each. Local search makes most of its gain in the first few
seconds and then levels off. CP-SAT gains more from each extra second, because
each chunk's solver has longer to search. With several workers its results vary
a little from run to run, so the curve is not always smooth.

In [ ]:
budgets = [5, 10, 20, 40, 80]
budget_results = {"Local search": [], "CP-SAT": []}
for budget in budgets:
    for label, cls, options in (
        ("Local search", LocalSearchScheduler, {"time_limit": budget, "seed": 1}),
        ("CP-SAT", CpSatScheduler, {"solver_time_limit": budget, "seed": 1}),
    ):
        scheduler = cls(config, targets, begin, end, **options)
        scheduler.schedule()
        budget_results[label].append(scheduler.score[0] / 3600)
    print(
        f"{budget:3d} s: local search {budget_results['Local search'][-1]:6.0f}, "
        f"CP-SAT {budget_results['CP-SAT'][-1]:6.0f} merit-hours"
    )

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4))
for label, scores in budget_results.items():
    ax.plot(budgets, scores, "o-", label=label, color=COLORS[label])
# Both start from the priority-first plan; its score is the baseline.
priority_score = schedulers["Local search"].initial_score[0] / 3600
ax.axhline(priority_score, color="grey", ls="--", label="Priority")
ax.set_xscale("log")
ax.minorticks_off()
ax.set_xticks(budgets, [str(b) for b in budgets])
ax.set_xlabel("Planning time budget (s)")
ax.set_ylabel("Merit-weighted science (merit-hours)")
ax.legend()
fig.tight_layout()
plt.show()

## Step 13: Benchmark in Rolling-Horizon Simulations

In operations the plan is rebuilt as the day goes on. Here we run the
Rolling-Horizon DITL example's setup (12-hour horizon, replanning every 6 hours,
15-minute commit lead time, and a ToO 8 hours in) with each scheduler on three
random sets of targets. Local search and CP-SAT each get 15 seconds per plan.

Each run is set up through the configuration's `scheduler` section: rolling mode,
the scheduler `kind` and its settings, and the replanning schedule.
`create_ditl` builds the `RollingHorizonDITL` it describes.

In [ ]:
# The Rolling-Horizon DITL example's replanning schedule.
REPLANNING = ReplanSettings(
    horizon_seconds=12 * 3600,
    replan_interval_seconds=6 * 3600,
    commit_lead_time_seconds=15 * 60,
    rapid_replans=True,
    allow_interrupts=True,
)


def run_rolling(seed: int, scheduler: SchedulerConfig) -> RollingHorizonDITL:
    config.scheduler = scheduler
    ditl = create_ditl(config, make_targets(seed), begin, end)
    ditl.submit_too(
        obsid=900001,
        ra=150.0,
        dec=30.0,
        merit=1000,
        exptime=600,
        name="GRB 251201A",
        submit_time=begin + timedelta(hours=8),
        deadline=begin + timedelta(hours=8, minutes=45),
    )
    ditl.calc()
    return ditl


ROLLING = {
    "Priority": SchedulerConfig(mode="rolling", replanning=REPLANNING),
    "Local search": SchedulerConfig(
        mode="rolling",
        kind="local_search",
        time_limit=15.0,
        seed=1,
        replanning=REPLANNING,
    ),
    "CP-SAT": SchedulerConfig(
        mode="rolling",
        kind="cp_sat",
        solver_time_limit=15.0,
        seed=1,
        replanning=REPLANNING,
    ),
}
seeds = [1, 2, 3]
results = {label: [] for label in ROLLING}
for seed in seeds:
    for label, scheduler in ROLLING.items():
        ditl = run_rolling(seed, scheduler)
        r = {
            "idle": idle_percent(ditl),
            "science": science_hours(ditl),
            "merit": merit_hours(ditl),
            "planning": sum(rec.planning_seconds for rec in ditl.replans),
            "mismatches": len(ditl.validate_plan_matches_execution()),
        }
        results[label].append(r)
        print(
            f"seed {seed}  {label:<13} idle {r['idle']:4.1f}%  "
            f"science {r['science']:5.2f} h  merit-weighted {r['merit']:5.0f}  "
            f"planning {r['planning']:5.1f} s  mismatches {r['mismatches']}"
        )

In [ ]:
x = np.arange(len(seeds))
labels = [f"seed {s}" for s in seeds]
priority_rows = results["Priority"]
compared = ["Local search", "CP-SAT"]
width = 0.38

fig, (idle_ax, science_ax, merit_ax, time_ax) = plt.subplots(1, 4, figsize=(13, 3.5))

# Idle time and planning time for every scheduler.
width3 = 0.8 / len(results)
for ax, key, title in (
    (idle_ax, "idle", "Idle (% of day)"),
    (time_ax, "planning", "Planning time (s)"),
):
    for n, (label, rows) in enumerate(results.items()):
        offset = (n - (len(results) - 1) / 2) * width3
        ax.bar(
            x + offset, [r[key] for r in rows], width3, label=label, color=COLORS[label]
        )
    ax.set_title(title)
idle_ax.legend(fontsize=8)

# Science and merit-weighted science as gains over the priority-first scheduler.
for n, label in enumerate(compared):
    offset = (n - 0.5) * width
    rows = results[label]
    science_ax.bar(
        x + offset,
        [r["science"] - p["science"] for p, r in zip(priority_rows, rows)],
        width,
        label=label,
        color=COLORS[label],
    )
    merit_ax.bar(
        x + offset,
        [100 * (r["merit"] / p["merit"] - 1) for p, r in zip(priority_rows, rows)],
        width,
        label=label,
        color=COLORS[label],
    )
science_ax.set_title("Science gained (h)")
merit_ax.set_title("Merit-weighted science gained (%)")

for ax in (idle_ax, science_ax, merit_ax, time_ax):
    ax.set_xticks(x, labels)
fig.tight_layout()
plt.show()

print(f"{'mean':<13} {'idle':>6} {'science':>8} {'merit-weighted':>15} {'planning':>9}")
for label, rows in results.items():
    mean = {key: np.mean([r[key] for r in rows]) for key in rows[0]}
    print(
        f"{label:<13} {mean['idle']:5.1f}% {mean['science']:7.2f}h "
        f"{mean['merit']:15.0f} {mean['planning']:8.1f}s"
    )

## Summary

* `CpSatScheduler` and `LocalSearchScheduler` both start from the priority-first plan
  and never return a worse one, and their plans execute exactly as planned.
* On these targets CP-SAT does clearly better than local search for the same
  planning time: it fills nearly all the idle time, and collects more science
  and more merit-weighted science, in a single plan and in rolling-horizon
  simulations.
* CP-SAT rebuilds a whole chunk at a time, so it finds rearrangements that local
  search, changing one snapshot at a time, reaches slowly or not at all. Local
  search can move snapshots anywhere in the horizon, while each CP-SAT chunk has
  to leave the rest of the priority-first plan room to follow it.
* CP-SAT keeps gaining from more solver time well after local search has
  levelled off. Splitting the same time between the solver and local search
  afterwards (`time_limit`) did a little worse than giving it all to the solver.
* Neither rewards completing a target, so with plenty of targets CP-SAT can spread
  its time more widely than the priority-first plan does.
* Use `workers=1` with a `seed` when runs must repeat exactly.

See the planning guide (Optimizing plans with CP-SAT) for how the model is built
and its limitations.